# Lab 2 — ข้างในสาย MCP (EP13)

Lab 1 เราใช้ `ClientSession` คุยกับ server — มันซ่อนข้อความทั้งหมดไว้
Lab นี้เราจะเป็น client เอง: เปิด server แล้ว **พิมพ์ข้อความ JSON-RPC 2.0 ทีละบรรทัด** ลงท่อ stdin และอ่านคำตอบดิบจาก stdout (§21.4)

**สิ่งที่จะได้ทำ**
1. เปิด server เป็นโปรแกรมลูก — transport แบบ stdio คือ JSON หนึ่งข้อความต่อหนึ่งบรรทัด
2. **Handshake:** `initialize` → ตกลงเวอร์ชันและความสามารถ → `notifications/initialized`
3. **Request / response:** `tools/list`, `tools/call` — จับคู่ด้วย `id`
4. **Error สองระดับ:** error ของ tool (`isError`) กับ error ของโปรโตคอล (`-32602`)
5. **Notification:** ความคืบหน้า (`notifications/progress`) และรายการ tool ที่เปลี่ยน (`list_changed`)
6. **Shutdown:** ปิดท่อ แล้ว server ปิดตัวเอง

Lab นี้ไม่เรียก model จึงไม่ต้องใช้ API key และไม่มีค่าใช้จ่าย

> อ้างอิง: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §21.2.3, §21.4, §21.5.2 — CC BY-SA 4.0

> ✏️ **Starter notebook — ฉบับให้ผู้เรียนลงมือเขียน**
>
> โค้ดส่วนใหญ่เตรียมไว้ให้แล้ว เขียนเพิ่มเฉพาะ **6 จุดที่มี `✏️ TODO`** (แทนที่ `___` หรือ `...` ด้วยโค้ดของคุณ) ทุกจุดบอกเวลาในวิดีโอที่อธิบายโค้ดตรงนั้น (▶ m:ss) — ดูคลิปก่อน แล้วลองเขียนเองโดยไม่เปิดเฉลย
>
> รันทุก cell ตามลำดับจากบนลงล่าง ถ้า cell ไหนยังเติมไม่ครบจะเจอ `NameError` หรือ `TypeError` เป็นเรื่องปกติ
>
> ติดตรงไหน เปิดเฉลยใน **`ep13-lab2-mcp-wire.ipynb`** (โค้ดชุดเดียวกับในวิดีโอ)

## 0. เตรียมเครื่องก่อนเริ่ม

| สิ่งที่ต้องมี | รายละเอียด |
|---|---|
| Python 3.10 ขึ้นไป | หรือใช้ Google Colab ซึ่งมีให้แล้ว |
| Packages | `mcp` (ใช้ฝั่ง server เท่านั้น — ฝั่ง client เราเขียนเอง) |

In [ ]:
%pip install -q "mcp==1.30.0"

## 1. Server ตัวเดิม + tool ใหม่สองตัว

`search_papers` และ resource `paper://{paper_id}` จาก Lab 1 · บวก tool ใหม่ที่ไว้ดูข้อความสองแบบ
- `index_library` — งานที่ใช้เวลา และ **รายงานความคืบหน้า** (§21.4.4)
- `enable_citations` — เพิ่ม tool `cite_paper` ระหว่าง session แล้ว **แจ้งว่ารายการ tool เปลี่ยน** (§21.5.2)

In [ ]:
%%writefile paper_server.py
import json, asyncio
from mcp.server.fastmcp import FastMCP, Context

mcp = FastMCP("cogito-papers")

PAPERS = {  # teaching data: real arXiv ids, titles and years; short summaries written for this lab
    "2211.14730": {"title": "A Time Series is Worth 64 Words: Long-term Forecasting with Transformers",
                   "authors": "Nie et al.", "year": 2023, "summary": "PatchTST: split each series into patches, one channel at a time"},
    "2310.06625": {"title": "iTransformer: Inverted Transformers Are Effective for Time Series Forecasting",
                   "authors": "Liu et al.", "year": 2024, "summary": "attention across variables instead of across time steps"},
    "2310.10688": {"title": "A decoder-only foundation model for time-series forecasting",
                   "authors": "Das et al.", "year": 2024, "summary": "TimesFM: one pretrained model that forecasts many series zero-shot"},
    "2012.07436": {"title": "Informer: Beyond Efficient Transformer for Long Sequence Time-Series Forecasting",
                   "authors": "Zhou et al.", "year": 2021, "summary": "sparse attention to make long forecasts cheaper"},
    "2205.13504": {"title": "Are Transformers Effective for Time Series Forecasting?",
                   "authors": "Zeng et al.", "year": 2023, "summary": "a simple linear model beats several transformers"},
}

@mcp.tool()
def search_papers(query: str, max_results: int = 3) -> str:
    """Search the paper library by keyword in the title or summary.
    Use this to find papers on a topic. Returns a JSON list of {id, title, year}, best match first."""
    words = query.lower().split()
    scored = []
    for pid, p in PAPERS.items():
        score = sum(w in (p["title"] + " " + p["summary"]).lower() for w in words)
        if score:
            scored.append((score, {"id": pid, "title": p["title"], "year": p["year"]}))
    scored.sort(key=lambda s: -s[0])
    return json.dumps([paper for _, paper in scored[:max_results]])

@mcp.resource("paper://{paper_id}")
def get_paper(paper_id: str) -> str:
    """One paper's full record, as JSON."""
    return json.dumps({"id": paper_id, **PAPERS[paper_id]})

In [ ]:
%%writefile -a paper_server.py

@mcp.tool()
async def index_library(ctx: Context) -> str:
    """Rebuild the search index. Slow; reports progress."""
    n = len(PAPERS)
    for i in range(1, n + 1):
        await asyncio.sleep(0.3)
        await ctx.report_progress(i, n, f"indexed {i}/{n} papers")
    return f"indexed {n} papers"

In [ ]:
%%writefile -a paper_server.py

@mcp.tool()
async def enable_citations(ctx: Context) -> str:
    """Unlock the cite_paper tool for this session."""
    def cite_paper(paper_id: str) -> str:
        """Format a short citation for one paper, by its arXiv id."""
        p = PAPERS[paper_id]
        return f'{p["authors"]} ({p["year"]}). {p["title"]}. arXiv:{paper_id}'
    mcp.add_tool(cite_paper)
    await ctx.session.send_tool_list_changed()
    return "cite_paper is now available"

if __name__ == "__main__":
    mcp.run()

## 2. เปิด server และต่อท่อเอง

transport แบบ stdio (§21.2.2): client เปิด server เป็นโปรแกรมลูก แล้วคุยผ่าน **stdin / stdout**
ข้อความหนึ่งข้อความ = JSON หนึ่งบรรทัด · log ของ server ออกทาง stderr (เราเก็บลงไฟล์)

In [ ]:
import sys, json, subprocess

server = subprocess.Popen([sys.executable, "paper_server.py"],
                          stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                          stderr=open("server.log", "w"), text=True)

In [ ]:
# ✏️ TODO 1 (▶ วิดีโอ 2:31): เขียนข้อความเป็น JSON หนึ่งบรรทัด (ปิดท้ายด้วยขึ้นบรรทัดใหม่)
def send(msg):
    server.stdin.write(___)
    server.stdin.flush()
    print("➡️ ", json.dumps(msg)[:300])

def recv():
    msg = json.loads(server.stdout.readline())
    print("⬅️ ", json.dumps(msg)[:300])
    return msg

## 3. Handshake: `initialize`

ข้อความแรกต้องเป็น `initialize` (§21.4.2): เวอร์ชันโปรโตคอล · ความสามารถของ client · ชื่อ client
เราส่งเวอร์ชัน `2024-11-05` ตามตัวอย่างในตำรา

In [ ]:
# ✏️ TODO 2 (▶ วิดีโอ 3:01): ตั้ง method ของคำขอแรก
send({"jsonrpc": "2.0", "id": 1, "method": ___,
      "params": {"protocolVersion": "2024-11-05", "capabilities": {},
                 "clientInfo": {"name": "by-hand", "version": "0.1"}}})
init = recv()
print("version:", init["result"]["protocolVersion"])
print("server capabilities:", init["result"]["capabilities"])

In [ ]:
# ✏️ TODO 3 (▶ วิดีโอ 3:42): แจ้งว่า client พร้อม (notification: ไม่มี id)
send({"jsonrpc": "2.0", "method": ___})

## 4. Request และ response

ทุกคำขอมี `id` · คำตอบกลับมาพร้อม `id` เดิม (§21.4.1) — เหมือนบัตรคิวร้านอาหาร

In [ ]:
send({"jsonrpc": "2.0", "id": 2, "method": "tools/list"})
tools = recv()["result"]["tools"]
for t in tools:
    print("🔧", t["name"], "· required:", t["inputSchema"].get("required", []))

In [ ]:
send({"jsonrpc": "2.0", "id": 42, "method": "tools/call",
      "params": {"name": "search_papers", "arguments": {"query": "linear", "max_results": 2}}})
reply = recv()
print("id:", reply["id"], "· isError:", reply["result"]["isError"])
print(reply["result"]["content"][0]["text"])

## 5. Error สองระดับ

| ระดับ | เกิดเมื่อ | หน้าตา |
|---|---|---|
| **tool** | tool ทำงานไม่สำเร็จ (argument ผิด, ข้อมูลไม่มี) | `result` ปกติ แต่ `isError: true` — ข้อความกลับไปให้ model อ่านได้ |
| **โปรโตคอล** | ตัวคำขอเองผิด | ไม่มี `result` · มี `error` พร้อมรหัส เช่น `-32602` invalid params (§21.4.3) |

In [ ]:
send({"jsonrpc": "2.0", "id": 43, "method": "tools/call",
      "params": {"name": "search_papers", "arguments": {}}})
bad_tool = recv()
print("has result:", "result" in bad_tool, "· isError:", bad_tool["result"]["isError"])

In [ ]:
send({"jsonrpc": "2.0", "id": 44, "method": "resources/read", "params": {}})
bad_request = recv()
print("has result:", "result" in bad_request, "· error:", bad_request["error"])

## 6. Notification: ความคืบหน้า และรายการ tool ที่เปลี่ยน

คำขอที่ใส่ `_meta.progressToken` → server ส่ง `notifications/progress` มาก่อนคำตอบ (§21.4.4)
เราจึงต้องอ่านไปเรื่อยๆ จนเจอข้อความที่มี `id` ของเรา

In [ ]:
# ✏️ TODO 4 (▶ วิดีโอ 5:28): หยุดเมื่อเจอข้อความที่ id ตรงกับคำขอ
def recv_until(request_id):
    while True:
        msg = recv()
        if ___:
            return msg

# ✏️ TODO 5 (▶ วิดีโอ 5:39): ขอรับความคืบหน้าด้วย progressToken

send({"jsonrpc": "2.0", "id": 45, "method": "tools/call",
      "params": {"name": "index_library", "arguments": {}, "_meta": ___}})
print(recv_until(45)["result"]["content"][0]["text"])

In [ ]:
send({"jsonrpc": "2.0", "id": 46, "method": "tools/call", "params": {"name": "enable_citations", "arguments": {}}})
recv_until(46)
send({"jsonrpc": "2.0", "id": 47, "method": "tools/list"})
print([t["name"] for t in recv_until(47)["result"]["tools"]])

## 7. Shutdown

ด้วย stdio client ปิดการเชื่อมต่อโดย **ปิดท่อ stdin** ของ server แล้วรอให้มันจบ

In [ ]:
# ✏️ TODO 6 (▶ วิดีโอ 6:29): ปิดท่อ stdin เพื่อสั่งปิด server
___
print("exit code:", server.wait(timeout=10))
print("server saw:", sorted({line.split("type ")[1].strip() for line in open("server.log") if "request of type" in line}))

## สรุป

| ข้อความ | สิ่งที่เห็น |
|---|---|
| `initialize` (id 1) | ตกลงเวอร์ชัน 2024-11-05 · server ประกาศ tools, resources, prompts |
| `notifications/initialized` | ไม่มี id · ไม่มีคำตอบ |
| `tools/call` (id 42) | คำตอบ id 42 · `content` + `isError: false` |
| tool ล้มเหลว (id 43) | `result` ปกติ + `isError: true` — ส่งให้ model อ่านได้ |
| คำขอผิด (id 44) | `error` รหัส `-32602` — ไม่มี `result` |
| `progressToken` (id 45) | `notifications/progress` 1/5 … 5/5 แล้วคำตอบ |
| `enable_citations` (id 46) | `notifications/tools/list_changed` → `tools/list` มี `cite_paper` |
| ปิด stdin | server จบด้วยรหัส 0 |

**แบบฝึกหัด**
1. ส่ง `tools/call` ก่อน `initialize` — server ตอบอะไร
2. ส่ง `resources/read` ของ `paper://2310.06625` เอง แล้วหา `contents` ในคำตอบ
3. ส่งคำขอสองคำขอ (id 50, 51) ติดกันก่อนอ่าน แล้วดูว่าคำตอบมาตามลำดับไหน — `id` ช่วยอะไร

**ถัดไป — Lab 3:** เขียน host ให้ agent ของเรา ต่อ server สองตัวพร้อมกัน และหยุดถามก่อนเรียก tool ที่ย้อนกลับไม่ได้